# Thermal-Contrast — TermoDataset + U-Net

Клип кадров → **contrast maps** (`features.py`) → static U-Net.

| preset | Каналы |
|--------|--------|
| `minimal` | max−min |
| `combo` | max−min, max−t₀, σ(t) |
| `full` | + PCA₁(t) |

Данные — только **TPU** через [`TermoDataset`](../../datasets/datasets.py) (`dataset_tpu`, `sample*.mat`).
Маски — **`labels/table_mask_raw/sample{n}.png`** (как в yaml-пайплайне).

> MPS: `NUM_WORKERS=0`, `COMPILE=False`.

In [ ]:
# === параметры ===
PRESET = "combo"              # minimal | delta | combo | pca | full
EPOCHS = 30
BATCH_SIZE = 8
TEST_EVERY = 4
NUM_WORKERS = 0
CLIP_LEN = 12
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 10.0
COMPILE = False
DEVICE = "auto"
RESUME = None                   # None | "best" | "last" | Path
PREVIEW_EVERY = 3
MAT_PATH = "data/sample11.mat"  # inference на произвольный .mat
MAT_KEY = "data"                # ключ в .mat (kaggle/tpu: "data")
THRESHOLD = 0.5

# TermoDataset: только TPU sample*.mat; маски → labels/table_mask_raw
TERMO_INCLUDE = ["dataset_tpu"]
TRANSFORM = None                # или callable(data_np, mask_np) -> (data, mask)

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import clear_output, display
from scipy.io import loadmat
from torch.utils.data import DataLoader

CHANNEL_TITLES = {
    "maxmin": "max − min",
    "maxfirst": "max − t₀",
    "minfirst": "min − t₀",
    "lastfirst": "last − t₀",
    "std": "σ(t)",
    "mean": "⟨t⟩",
    "pca1": "PCA₁(t)",
}


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    bases = [start, start.parent] if start.name == "notebooks" else [start]
    seen: set[Path] = set()
    for base in bases:
        for p in [base, *base.parents]:
            p = p.resolve()
            if p in seen:
                continue
            seen.add(p)
            if (p / "datasets" / "datasets.py").is_file() and (p / "data").is_dir():
                return p
    raise FileNotFoundError(f"корень репо не найден, cwd={start}")


ROOT = find_project_root()
TC = ROOT / "models" / "Thermal-Contrast"
SEG = ROOT / "models"
if str(TC) not in sys.path:
    sys.path.insert(0, str(TC))
for p in (SEG, ROOT):
    sp = str(p)
    if sp not in sys.path:
        sys.path.append(sp)

from common.data import INPUT_SIZE, _center_crop
from common.device import get_device
from common.loop import run_epoch
from common.metrics import BCEDiceLoss, dice_score, iou_score
from common.mps_train import (
    load_model_weights,
    optimize_model_mps,
    setup_mps_env,
    suggest_num_workers,
)
from common.split import split_videos
from common.termo_data import TermoIndexDataset, _restrict_videos
from common.tracking import HistoryTracker
from data import ContrastTermoDataset
from datasets import TermoDataset
from features import PRESETS, channel_names, collapse_temporal, normalize_contrast
from model import UNetModel

setup_mps_env()
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

device = get_device(DEVICE)
nw = suggest_num_workers(device) if NUM_WORKERS is None else NUM_WORKERS
termo_root = ROOT / "datasets" / "datasets_list"
channels = channel_names(PRESET)

print(f"ROOT={ROOT.name}  device={device}  preset={PRESET}")
print(f"channels={channels}  clip_len={CLIP_LEN}  bs={BATCH_SIZE}")

## 1. TermoDataset

In [ ]:
termo_ds = TermoDataset(
    root_dir=str(termo_root),
    include=TERMO_INCLUDE,
    transform=TRANSFORM,
)

print(f"TermoDataset: {len(termo_ds)} videos")
print(f"include={TERMO_INCLUDE!r}  transform={TRANSFORM}")
mp0 = Path(termo_ds.items[0][1])
print(f"sample[0]: mat={Path(termo_ds.items[0][0]).name}  mask={mp0.as_posix()}")
print(f"shape sample[0]: data={termo_ds[0][0].shape}  mask={termo_ds[0][1].shape}  gt_pos={termo_ds[0][1].gt(0).float().mean():.3f}")

base = TermoIndexDataset(termo_ds)
train_ids, test_ids = split_videos(list(base.video_ids), test_every=TEST_EVERY)
train_base = _restrict_videos(base, train_ids)
test_base = _restrict_videos(base, test_ids)

kw = dict(preset=PRESET, clip_len=CLIP_LEN, size=INPUT_SIZE)
train_ds = ContrastTermoDataset(train_base, random_clip=True, **kw)
test_ds = ContrastTermoDataset(test_base, random_clip=False, **kw)

loader_kw = dict(num_workers=nw, pin_memory=False, persistent_workers=bool(nw))
if nw > 0:
    loader_kw["prefetch_factor"] = 3

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_kw
)

print(
    f"train {len(train_ds.video_ids)} vid / {len(train_ds)} samples\n"
    f"test  {len(test_ds.video_ids)} vid / {len(test_ds)} samples\n"
    f"test ids: {test_ds.video_ids[:6]}{'…' if len(test_ds.video_ids) > 6 else ''}"
)

## 2. Каналы — как клип сворачивается

In [ ]:
def show_collapse_pipeline(ds: ContrastTermoDataset, idx: int = 0, *, max_frames: int = 8) -> None:
    vid = ds.video_ids[idx]
    frames, mask = ds.temporal[idx]
    raw = frames[:, 0].numpy()
    T = raw.shape[0]
    pick = np.linspace(0, T - 1, num=min(max_frames, T), dtype=int)

    raw_feat = collapse_temporal(raw, ds.channels)
    feat_np = normalize_contrast(raw_feat)
    mask_np = mask.squeeze().numpy()
    n_ch = len(ds.channels)

    fig1, ax1 = plt.subplots(figsize=(14, 2.2))
    strip = np.hstack([raw[t] for t in pick])
    ax1.imshow(strip, cmap="inferno", aspect="auto")
    ax1.set_title(f"клип T={T}, video={vid!r}")
    ax1.axis("off")
    fig1.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(2, n_ch + 1, figsize=(2.6 * (n_ch + 1), 5))
    for c, name in enumerate(ds.channels):
        ax2[0, c].imshow(raw_feat[c], cmap="inferno")
        ax2[0, c].set_title(f"{CHANNEL_TITLES.get(name, name)} raw")
        ax2[0, c].axis("off")
        ax2[1, c].imshow(feat_np[c], cmap="magma")
        ax2[1, c].set_title("norm")
        ax2[1, c].axis("off")
    ax2[0, -1].axis("off")
    ax2[1, -1].imshow(mask_np, cmap="gray", vmin=0, vmax=1)
    ax2[1, -1].set_title(f"GT pos={mask_np.mean():.3f}")
    ax2[1, -1].axis("off")
    fig2.suptitle("collapse → U-Net input (norm = per-channel)", fontsize=11)
    fig2.tight_layout()
    plt.show()


show_collapse_pipeline(train_ds, 0)
if len(test_ds):
    show_collapse_pipeline(test_ds, 0)

## 3. Обучение

In [ ]:
def plot_training_dashboard(history, *, epoch, end_epoch, best_iou, best_epoch, title=""):
    if not history:
        return
    ep = [float(r["epoch"]) for r in history]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(title or f"Thermal-Contrast [{PRESET}]  epoch {epoch}/{end_epoch}", fontsize=12)
    for ax, (tk, vk, ylab) in zip(
        axes.flat[:3],
        [("train_loss", "test_loss", "Loss"), ("train_iou", "test_iou", "IoU"), ("train_dice", "test_dice", "Dice")],
    ):
        ax.plot(ep, [float(r[tk]) for r in history], "o-", label="train", ms=4)
        ax.plot(ep, [float(r[vk]) for r in history], "s-", label="test", ms=4)
        if float(best_epoch) in ep:
            ax.axvline(best_epoch, color="green", ls="--", alpha=0.4)
        ax.set_title(ylab)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    ax = axes[1, 1]
    ax.axis("off")
    last = history[-1]
    ax.text(
        0.05, 0.95,
        f"epoch {epoch}/{end_epoch}\n"
        f"test IoU {last['test_iou']:.4f}  Dice {last['test_dice']:.4f}\n"
        f"best IoU {best_iou:.4f} @ {best_epoch}\n"
        f"ch={train_ds.channels}",
        va="top", family="monospace", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3),
    )
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def preview_prediction(model, ds, device, *, idx=0, title=""):
    model.eval()
    feat, mask = ds[idx]
    with torch.no_grad():
        prob = torch.sigmoid(model(feat.unsqueeze(0).to(device))).squeeze().cpu().numpy()
    pred = (prob > THRESHOLD).astype(np.float32)
    gt = mask.squeeze().numpy()
    d, iou = dice_score(pred, gt), iou_score(pred, gt)
    n = feat.shape[0]
    fig, axes = plt.subplots(2, max(n, 3), figsize=(2.4 * max(n, 3), 5))
    for c in range(n):
        axes[0, c].imshow(feat[c].numpy(), cmap="magma")
        axes[0, c].set_title(CHANNEL_TITLES.get(ds.channels[c], ds.channels[c]), fontsize=9)
        axes[0, c].axis("off")
    for c in range(n, axes.shape[1]):
        axes[0, c].axis("off")
    axes[1, 0].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title("GT")
    axes[1, 0].axis("off")
    axes[1, 1].imshow(prob, cmap="magma", vmin=0, vmax=1)
    axes[1, 1].set_title("prob")
    axes[1, 1].axis("off")
    axes[1, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, 2].set_title(f"pred Dice={d:.3f}")
    axes[1, 2].axis("off")
    for c in range(3, axes.shape[1]):
        axes[1, c].axis("off")
    if title:
        fig.suptitle(f"{title}  IoU={iou:.3f}", fontsize=10)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def train_termo(
    *, epochs, device, resume=None, on_epoch_end=None,
) -> tuple[HistoryTracker, nn.Module, float, int]:
    model = UNetModel(in_channels=train_ds.num_channels, num_classes=1)
    ckpt_best = TC / f"model_contrast_{PRESET}_best.tar"
    ckpt_last = TC / f"model_contrast_{PRESET}_last.tar"

    resume_path = None
    if resume is not None:
        resume_path = (
            ckpt_best if str(resume) == "best" else ckpt_last if str(resume) == "last" else Path(resume)
        )
        load_model_weights(model, resume_path)

    use_cl = device.type == "cuda"
    model = optimize_model_mps(model, device, channels_last=use_cl, compile_model=COMPILE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=LR * 0.05)
    criterion = BCEDiceLoss(pos_weight=POS_WEIGHT)
    tracker = HistoryTracker(TC / "runs" / PRESET, tag=f"contrast-{PRESET}", resume=resume_path is not None)

    start = tracker.next_epoch if resume_path else 1
    end = start + epochs - 1
    best_iou, best_epoch = tracker.best_test_iou() if resume_path else (-1.0, -1)

    print(
        f"\n=== train TermoDataset | {device} | in_ch={train_ds.num_channels} {train_ds.channels} ===\n"
        f"train {len(train_ds.video_ids)} vid / {len(train_ds)}  "
        f"test {len(test_ds.video_ids)} vid / {len(test_ds)}"
    )

    for epoch in range(start, end + 1):
        train_loss, train_m = run_epoch(
            model, train_loader, device, opt,
            desc=f"train {epoch}", channels_last=use_cl, loss_fn=criterion, grad_clip=1.0,
        )
        test_loss, test_m = run_epoch(
            model, test_loader, device,
            desc=f"test {epoch}", channels_last=use_cl, loss_fn=criterion,
        )
        sched.step()
        row = tracker.log(epoch, train_loss, test_loss, train_m, test_m)
        print(tracker.format_line(row, end))
        if test_m["iou"] > best_iou:
            best_iou, best_epoch = test_m["iou"], epoch
            torch.save(model.state_dict(), ckpt_best)
            print(f"  >>> best test IoU {best_iou:.4f} @ {best_epoch}")
        if on_epoch_end is not None:
            on_epoch_end(tracker, row, epoch, end, best_iou, best_epoch, opt, model)

    torch.save(model.state_dict(), ckpt_last)
    print(f"done. best IoU {best_iou:.4f} @ {best_epoch}")
    return tracker, model, best_iou, best_epoch


def _on_epoch_end(tracker, row, epoch, end_epoch, best_iou, best_epoch, opt, model):
    clear_output(wait=True)
    print(tracker.format_line(row, end_epoch), f"| best IoU {best_iou:.4f}")
    plot_training_dashboard(
        tracker.history, epoch=epoch, end_epoch=end_epoch,
        best_iou=best_iou, best_epoch=best_epoch,
    )
    if PREVIEW_EVERY and epoch % PREVIEW_EVERY == 0 and len(test_ds):
        preview_prediction(model, test_ds, device, idx=0, title=f"test[0] @ {epoch}")

In [ ]:
tracker, model, best_iou, best_epoch = train_termo(
    epochs=EPOCHS,
    device=device,
    resume=RESUME,
    on_epoch_end=_on_epoch_end,
)

plot_training_dashboard(
    tracker.history,
    epoch=int(tracker.history[-1]["epoch"]),
    end_epoch=int(tracker.history[-1]["epoch"]),
    best_iou=best_iou,
    best_epoch=best_epoch,
    title=f"FINAL — {PRESET}",
)

## 4. Test-сэмплы

In [ ]:
for i in range(len(test_ds)):
    preview_prediction(
        model, test_ds, device, idx=i,
        title=f"test[{i}] {test_ds.video_ids[i]}",
    )

## 5. Inference на `.mat`

In [ ]:
def load_mat_clip(
    path: str | Path,
    *,
    mat_key: str = MAT_KEY,
    clip_len: int = CLIP_LEN,
    size: int = INPUT_SIZE,
) -> np.ndarray:
    """Тот же пайплайн, что TermoDataset + центральный клип."""
    import torch.nn.functional as F

    data = TermoDataset._load_mat_volume(str(path), mat_key)
    t = torch.from_numpy(data)
    if t.ndim == 3:
        t = t.permute(2, 0, 1)
    else:
        t = t.unsqueeze(0)
    t = F.interpolate(
        t.unsqueeze(0), size=(size, size), mode="bilinear", align_corners=False
    ).squeeze(0)
    t = (t - t.mean()) / (t.std() + 1e-8)

    n = min(clip_len, t.shape[0])
    t0 = max(0, (t.shape[0] - n) // 2)
    clip = t[t0 : t0 + n]
    return torch.stack([_center_crop(clip[i].float(), size) for i in range(n)]).numpy()


@torch.no_grad()
def predict_mat(model, mat_path, device, *, channels=None, threshold=THRESHOLD):
    model.eval()
    ch = channel_names(PRESET, channels=channels)
    raw = load_mat_clip(mat_path)
    feat = normalize_contrast(collapse_temporal(raw, ch))
    x = torch.from_numpy(feat).float().unsqueeze(0).to(device)
    prob = torch.sigmoid(model(x)).squeeze().cpu().numpy()
    pred = (prob > threshold).astype(np.float32)
    return {"clip": raw, "feat": feat, "prob": prob, "pred": pred, "channels": ch}


def show_mat_prediction(out, *, title=""):
    ch, feat, prob, pred = out["channels"], out["feat"], out["prob"], out["pred"]
    n = len(ch)
    fig, axes = plt.subplots(2, max(n, 3), figsize=(2.5 * max(n, 3), 5))
    for i, name in enumerate(ch):
        axes[0, i].imshow(feat[i], cmap="magma")
        axes[0, i].set_title(CHANNEL_TITLES.get(name, name))
        axes[0, i].axis("off")
    for i in range(n, axes.shape[1]):
        axes[0, i].axis("off")
    axes[1, 0].imshow(prob, cmap="magma", vmin=0, vmax=1)
    axes[1, 0].set_title("prob")
    axes[1, 0].axis("off")
    axes[1, 1].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, 1].set_title("pred")
    axes[1, 1].axis("off")
    axes[1, 2].imshow(out["clip"][len(out["clip"]) // 2], cmap="inferno")
    axes[1, 2].set_title("clip mid frame")
    axes[1, 2].axis("off")
    for i in range(3, axes.shape[1]):
        axes[1, i].axis("off")
    if title:
        fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()


mat_path = ROOT / MAT_PATH
if not mat_path.is_file():
    raise FileNotFoundError(mat_path)

out = predict_mat(model, mat_path, device)
show_mat_prediction(out, title=mat_path.name)
print(f"pred pos={out['pred'].mean():.4f}  channels={out['channels']}")